# Brain Decoding — Motif4Limbs — Grid Search

**Goal:** Find the best combination of:
- **Signal extraction config** — which TR(s) around the HRF peak to use as the trial's brain pattern
- **Classifier + regularisation** — Linear SVM, Logistic Regression, Ridge
- **Classification problem** — 4-class (all limbs) / hand vs foot / left vs right

All combinations are evaluated with **leave-one-run-out** cross-validation.  
Results are displayed as heatmaps. The best configuration gets a full analysis at the end.

---
### What is one sample?

For each trial, we extract the BOLD brain image at the HRF peak (~5s after onset).  
`NiftiMasker` flattens the 3D brain → one row of voxel values.  

```
             voxel_1  voxel_2  ...  voxel_N   →  label
Trial  1  [   0.42    -0.11   ...   0.05   ]  →  pied_droit
Trial  2  [   0.18     0.77   ...   1.12   ]  →  main_gauche
...
```


In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import nibabel as nib
from pathlib import Path
import time

try:
    from nilearn.maskers import NiftiMasker
except ImportError:
    from nilearn.input_data import NiftiMasker

from nilearn import plotting

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.base import clone

import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
# --- Settings ---
SUB   = '07'
SPACE = 'MNI152NLin2009cAsym'
TR    = 1.6   # seconds

# All HRF offsets we will sample (seconds after trial onset)
# We extract all of them once, then each config picks a subset
OFFSETS = [3.0, 4.0, 5.0, 6.0, 7.0, 8.0]

# Timing constants (sequence method)
BLANK_S = 1.0
STIM_S  = 2.2
PAUSE_S = 4.8

# Paths
FMRIPREP = Path('/neurospin/motif-stroke/7T_protocol/pilots/derivatives/fmriprep')
BEH_ROOT = Path('/neurospin/motif-stroke/7T_protocol/pilots/Behavioral_data')
SEQ_DIR  = BEH_ROOT / f'sub-{SUB}' / '4limbs' / 'seq'
FUNC_DIR = FMRIPREP / f'sub-{SUB}' / 'func'

print(f'Subject : sub-{SUB}  |  TR = {TR}s  |  Space = {SPACE}')
print(f'Offsets to extract: {OFFSETS} s')

---
## Step 1 — Load BOLD + extract multi-offset patterns

For each trial, we extract **6 brain volumes** (one per offset in OFFSETS).  
After masking, we get `X_all` of shape `(n_trials, n_offsets, n_voxels)`.  
The grid search then selects and averages the right offsets per configuration.


In [ ]:
def load_sequence_events(run_int):
    seq = pd.read_csv(SEQ_DIR / f'stim_sequence_run-{run_int}.csv').sort_values(['block_id', 'id'])
    rows, current, current_block = [], 0.0, None
    for _, r in seq.iterrows():
        if current_block is None:
            current_block = r['block_id']
        elif r['block_id'] != current_block:
            current += PAUSE_S
            current_block = r['block_id']
        current += BLANK_S
        rows.append({'onset': current, 'trial_type': r['block_name']})
        current += STIM_S
    return pd.DataFrame(rows)


# --- Extract all volumes ---
# We build a flat list: [trial0_offset0, trial0_offset1, ..., trial1_offset0, ...]
# so that NiftiMasker.fit_transform() can process them all in one call.

all_vols   = []   # flat list of 3D Nifti images
labels_raw = []   # condition label per trial
run_ids    = []   # run number per trial

for run in [1, 2]:
    pattern = f'sub-{SUB}_task-motif4limbs_dir-*_run-{run}_space-{SPACE}_desc-preproc_bold.nii.gz'
    bold_files = sorted(FUNC_DIR.glob(pattern))
    if not bold_files:
        print(f'  WARNING: no BOLD for run {run}')
        continue

    print(f'Loading run {run} BOLD...', end=' ', flush=True)
    bold_img  = nib.load(str(bold_files[0]))
    bold_data = bold_img.get_fdata()       # 4D array (x, y, z, n_TRs)
    n_trs     = bold_data.shape[-1]
    print(f'shape {bold_img.shape}  ({n_trs} TRs)')

    events = load_sequence_events(run)

    for _, trial in events.iterrows():
        # One 3D volume per offset
        for offset in OFFSETS:
            tr_idx = int(round((trial['onset'] + offset) / TR))
            tr_idx = max(0, min(tr_idx, n_trs - 1))
            vol = nib.Nifti1Image(bold_data[..., tr_idx].copy(), bold_img.affine)
            all_vols.append(vol)

        labels_raw.append(trial['trial_type'])
        run_ids.append(run)

labels  = np.array(labels_raw)
run_ids = np.array(run_ids)
n_trials  = len(labels)
n_offsets = len(OFFSETS)

print(f'\nTrials extracted : {n_trials}  ({n_trials * n_offsets} volumes total)')
print(f'Classes          : {np.unique(labels)}')

In [ ]:
# --- Apply brain mask once ---
mask_pattern = f'sub-{SUB}_task-motif4limbs_dir-*_run-1_space-{SPACE}_desc-brain_mask.nii.gz'
mask_path    = str(sorted(FUNC_DIR.glob(mask_pattern))[0])

print('Applying NiftiMasker to all volumes...', flush=True)
t0 = time.time()

masker = NiftiMasker(mask_img=mask_path, standardize=True, verbose=0)
X_flat = masker.fit_transform(all_vols)   # (n_trials * n_offsets, n_voxels)

# Reshape to (n_trials, n_offsets, n_voxels)
n_voxels = X_flat.shape[1]
X_all    = X_flat.reshape(n_trials, n_offsets, n_voxels)

print(f'Done in {time.time()-t0:.0f}s')
print(f'X_all shape: {X_all.shape}  (trials × offsets × voxels)')

---
## Step 2 — Define all configurations

### Signal extraction configs
Which TR(s) to use as the brain pattern for each trial?

| Config | What it does |
|---|---|
| `TR+Xs` | Single TR at onset + X seconds |
| `mean[a-b]` | Average all TRs between onset+a and onset+b seconds |

### Classifiers
All are linear (suitable for high-dimensional voxel spaces).  
Regularisation strength (C / alpha) controls overfitting.

### Problems
- **4-class**: distinguish all 4 limbs (chance = 25%)
- **hand vs foot**: hands vs feet (chance = 50%) — easy, large somatotopic distance
- **left vs right**: left body side vs right body side (chance = 50%) — harder, requires lateralisation


In [ ]:
# --- Signal extraction configs ---
# Each entry maps offset values → will be averaged across those offsets
signal_configs = {
    'TR+3s':       [3.0],
    'TR+4s':       [4.0],
    'TR+5s':       [5.0],
    'TR+6s':       [6.0],
    'TR+7s':       [7.0],
    'TR+8s':       [8.0],
    'mean[4-6s]':  [4.0, 5.0, 6.0],
    'mean[5-7s]':  [5.0, 6.0, 7.0],
    'mean[4-7s]':  [4.0, 5.0, 6.0, 7.0],
    'mean[3-8s]':  [3.0, 4.0, 5.0, 6.0, 7.0, 8.0],
}

# --- Classifiers ---
classifiers = {
    'LinSVC C=0.001': LinearSVC(C=0.001, max_iter=3000),
    'LinSVC C=0.01':  LinearSVC(C=0.01,  max_iter=3000),
    'LinSVC C=0.1':   LinearSVC(C=0.1,   max_iter=3000),
    'LinSVC C=1':     LinearSVC(C=1.0,   max_iter=3000),
    'LogReg C=0.01':  LogisticRegression(C=0.01, max_iter=2000, solver='lbfgs'),
    'LogReg C=0.1':   LogisticRegression(C=0.1,  max_iter=2000, solver='lbfgs'),
    'Ridge a=1':      RidgeClassifier(alpha=1.0),
    'Ridge a=10':     RidgeClassifier(alpha=10.0),
    'Ridge a=100':    RidgeClassifier(alpha=100.0),
}

# --- Problems ---
problems = {
    '4-class':      labels,
    'hand_vs_foot': np.where(np.isin(labels, ['main_droite', 'main_gauche']), 'hand', 'foot'),
    'left_vs_right': np.where(np.isin(labels, ['main_gauche', 'pied_gauche']), 'left', 'right'),
}
chance = {'4-class': 0.25, 'hand_vs_foot': 0.50, 'left_vs_right': 0.50}

n_total = len(signal_configs) * len(classifiers) * len(problems)
print(f'Signal configs : {len(signal_configs)}')
print(f'Classifiers    : {len(classifiers)}')
print(f'Problems       : {len(problems)}')
print(f'Total runs     : {n_total}')

---
## Step 3 — Grid search (runs overnight)


In [ ]:
logo    = LeaveOneGroupOut()
results = {}   # key: (problem, signal_config, classifier) → accuracy

t_start = time.time()
done    = 0

for prob_name, y_prob in problems.items():
    for sig_name, offsets_used in signal_configs.items():
        # Build X for this signal config by averaging the selected offsets
        indices = [OFFSETS.index(o) for o in offsets_used]
        X_sig   = X_all[:, indices, :].mean(axis=1)   # (n_trials, n_voxels)

        for clf_name, clf in classifiers.items():
            y_pred = cross_val_predict(
                clone(clf), X_sig, y_prob,
                groups=run_ids, cv=logo
            )
            acc = accuracy_score(y_prob, y_pred)
            results[(prob_name, sig_name, clf_name)] = acc

            done += 1
            elapsed = time.time() - t_start
            eta     = elapsed / done * (n_total - done)
            print(
                f'[{done:3d}/{n_total}]  {prob_name:12s} | '
                f'{sig_name:12s} | {clf_name:18s} | '
                f'acc={acc:.1%}  (ETA {eta:.0f}s)',
                flush=True
            )

print(f'\nGrid search complete in {time.time()-t_start:.0f}s')

---
## Step 4 — Results heatmaps

One heatmap per problem. Rows = classifiers, columns = signal configs.  
Colour scale starts at chance level.


In [ ]:
sig_names = list(signal_configs.keys())
clf_names = list(classifiers.keys())

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, prob_name in zip(axes, problems.keys()):
    mat = np.zeros((len(clf_names), len(sig_names)))
    for i, clf_n in enumerate(clf_names):
        for j, sig_n in enumerate(sig_names):
            mat[i, j] = results[(prob_name, sig_n, clf_n)]

    # Mark the best cell
    best_i, best_j = np.unravel_index(mat.argmax(), mat.shape)

    vmin = chance[prob_name]
    im   = ax.imshow(mat, vmin=vmin, vmax=1.0, cmap='RdYlGn', aspect='auto')

    ax.set_xticks(range(len(sig_names)))
    ax.set_xticklabels(sig_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(clf_names)))
    ax.set_yticklabels(clf_names, fontsize=9)
    ax.set_title(f'{prob_name}\nchance={vmin:.0%}  best={mat.max():.1%}', fontsize=11)

    # Cell values + highlight best
    for i in range(len(clf_names)):
        for j in range(len(sig_names)):
            color = 'black'
            weight = 'normal'
            if i == best_i and j == best_j:
                color  = 'navy'
                weight = 'bold'
                ax.add_patch(plt.Rectangle(
                    (j - 0.5, i - 0.5), 1, 1,
                    fill=False, edgecolor='navy', linewidth=3
                ))
            ax.text(j, i, f'{mat[i,j]:.0%}',
                    ha='center', va='center', fontsize=8,
                    color=color, fontweight=weight)

    plt.colorbar(im, ax=ax, fraction=0.03)

plt.suptitle(f'Decoding accuracy grid — sub-{SUB} | leave-one-run-out CV', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('decoding_grid_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: decoding_grid_results.png')

In [ ]:
# --- Summary: top 5 configs per problem ---
print('=' * 70)
for prob_name in problems:
    sub_results = {k: v for k, v in results.items() if k[0] == prob_name}
    ranked = sorted(sub_results.items(), key=lambda x: x[1], reverse=True)
    print(f'\n{prob_name.upper()}  (chance={chance[prob_name]:.0%})')
    print(f'{"Rank":<5} {"Signal config":<14} {"Classifier":<20} {"Accuracy"}')
    print('-' * 55)
    for rank, ((_, sig_n, clf_n), acc) in enumerate(ranked[:5], 1):
        print(f'{rank:<5} {sig_n:<14} {clf_n:<20} {acc:.1%}')
print('=' * 70)

---
## Step 5 — Full analysis on best configuration (4-class)


In [ ]:
# Find the best config for the 4-class problem
best_key  = max(
    [(k, v) for k, v in results.items() if k[0] == '4-class'],
    key=lambda x: x[1]
)
_, best_sig, best_clf = best_key[0]
best_acc = best_key[1]

print(f'Best 4-class config:')
print(f'  Signal  : {best_sig}')
print(f'  Model   : {best_clf}')
print(f'  Accuracy: {best_acc:.1%}  (chance=25%)')

# Rebuild X and get predictions
indices_best = [OFFSETS.index(o) for o in signal_configs[best_sig]]
X_best       = X_all[:, indices_best, :].mean(axis=1)
clf_best     = clone(classifiers[best_clf])

y_pred_best  = cross_val_predict(
    clf_best, X_best, labels, groups=run_ids, cv=logo
)

conditions = ['main_droite', 'main_gauche', 'pied_droit', 'pied_gauche']
label_fr   = {'main_droite': 'Right hand', 'main_gauche': 'Left hand',
               'pied_droit':  'Right foot', 'pied_gauche': 'Left foot'}

In [ ]:
# --- Confusion matrix ---
cm   = confusion_matrix(labels, y_pred_best, labels=conditions)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[label_fr[c] for c in conditions]
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(
    f'Confusion matrix — best 4-class config\n'
    f'{best_sig}  |  {best_clf}  |  acc={best_acc:.1%}'
)
plt.tight_layout()
plt.savefig('decoding_confusion_best.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Per-condition accuracy ---
print('Per-condition accuracy (best config):')
for cond in conditions:
    idx  = labels == cond
    acc  = accuracy_score(labels[idx], y_pred_best[idx])
    n    = idx.sum()
    print(f'  {label_fr[cond]:12s}: {acc:.1%}  ({n} trials)')

In [ ]:
# --- Mean brain patterns (best signal config) ---
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, cond in zip(axes.flat, conditions):
    mean_vec = X_best[labels == cond].mean(axis=0)
    mean_img = masker.inverse_transform(mean_vec)
    plotting.plot_stat_map(
        mean_img, axes=ax,
        title=f'{label_fr[cond]}  ({best_sig})',
        display_mode='z', cut_coords=5,
        colorbar=True, threshold=0.3,
    )

plt.suptitle(f'Mean BOLD pattern per limb — sub-{SUB}', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('decoding_mean_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Decoding weight maps (fit on all data) ---
clf_best.fit(X_best, labels)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (i, cond) in zip(axes.flat, enumerate(clf_best.classes_)):
    w     = clf_best.coef_[i]
    w_img = masker.inverse_transform(w)
    plotting.plot_stat_map(
        w_img, axes=ax,
        title=f'SVM weights — {label_fr.get(cond, cond)}',
        display_mode='z', cut_coords=5,
        colorbar=True,
        threshold=np.percentile(np.abs(w), 90),
    )

plt.suptitle(
    f'Per-class decoding weights — {best_clf}  |  {best_sig}\n'
    f'Red = votes FOR this limb  |  Blue = votes AGAINST',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('decoding_weights_per_class.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Overall discriminability map (mean |weight| across classes) ---
overall_w   = np.abs(clf_best.coef_).mean(axis=0)
overall_img = masker.inverse_transform(overall_w)

plotting.plot_stat_map(
    overall_img,
    title=f'Overall discriminability — top 10% most informative voxels\n{best_clf}  |  {best_sig}  |  acc={best_acc:.1%}',
    threshold=np.percentile(overall_w, 90),
    display_mode='z', cut_coords=8,
    colorbar=True,
)
plt.savefig('decoding_overall_weights.png', dpi=120, bbox_inches='tight')
plt.show()

print('Expected: bright voxels in lateral M1 (hands) and medial M1/SMA (feet)')

---
## Step 6 — Accuracy comparison across all problems


In [ ]:
# --- Bar chart: best accuracy per problem vs chance ---
fig, ax = plt.subplots(figsize=(8, 5))

prob_names   = list(problems.keys())
best_accs    = []
best_configs = []

for prob_name in prob_names:
    sub = {k: v for k, v in results.items() if k[0] == prob_name}
    best = max(sub.items(), key=lambda x: x[1])
    best_accs.append(best[1])
    best_configs.append(f"{best[0][1]}\n{best[0][2]}")

chance_vals = [chance[p] for p in prob_names]
x = np.arange(len(prob_names))
w = 0.35

bars1 = ax.bar(x - w/2, chance_vals, w, label='Chance', color='lightgray', edgecolor='gray')
bars2 = ax.bar(x + w/2, best_accs,   w, label='Best accuracy', color='steelblue', edgecolor='navy')

for bar, acc, cfg in zip(bars2, best_accs, best_configs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(prob_names, fontsize=11)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.set_title(f'Best decoding accuracy per problem — sub-{SUB}')
ax.legend()
ax.axhline(1.0, color='black', linestyle='--', alpha=0.3, label='Perfect')

# Print best config under each bar
for xi, cfg in zip(x + w/2, best_configs):
    ax.text(xi, -0.07, cfg, ha='center', va='top', fontsize=7, color='navy',
            transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig('decoding_summary.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Notes for interpretation

### What to look for

| Problem | Chance | Expected range | Why |
|---|---|---|---|
| 4-class | 25% | 50–80% | 4 limbs have distinct M1 representations at 7T |
| hand vs foot | 50% | 80–99% | Somatotopy is very clear, large spatial distance |
| left vs right | 50% | 55–75% | Requires contralateral lateralisation — harder |

### Interpreting the heatmap
- **Rows (classifiers):** Ridge is usually very stable; LinearSVC needs careful C tuning
- **Columns (signal configs):** averaging 3–5 TRs around the peak usually beats single TR (less noise)
- **Best cell** is outlined in navy

### Limitations
- Only 2 runs → LORO gives 2 test folds (accuracy estimate is noisy)
- No motion confound removal — could inflate accuracy if movement correlates with condition
- Whole-brain mask — ROI (motor cortex) would give cleaner results
